In [5]:
"""
Section 11: User-Affected Item Local Repair（矩阵化版本）
=============================================
针对 |U_T| 较大（~889）的情况，所有 loss 计算全部向量化，
避免 Python for-loop 逐 pair 迭代。
=============================================
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ─────────────────────────────────────────────────────────────
# 路径
# ─────────────────────────────────────────────────────────────
DATA_PATH      = "/Users/yubinhao/ml-1m/Amazon Review/数据集/Amazon_Reviews_Processed"
EMBED_PATH     = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/模型文件/embeddings_best.pt"
BASIS_DIR      = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_basis"
INFLUENCE_PATH = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_influence/influence_scores.pt"
UNLEARN_PATH   = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/遗忘实验/embeddings_unlearned.pt"
SAVE_DIR       = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/local_repair"
os.makedirs(SAVE_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 超参数
# ─────────────────────────────────────────────────────────────
W_PATTERN   = 0.8
W_TARGET    = 0.65
W_HISTORY   = 1.35
W_TOP       = 0.8
W_COLLAT    = 1.0
W_USER_REG  = 0.08
W_ITEM_REG  = 0.75

MARGIN      = 2
TEMPERATURE = 0.5

LR          = 0.025
EPOCHS      = 200
LOG_EVERY   = 20

TOP_K            = 8
LAMBDA_SCHEDULE  = [1.0, 0.9, 0.8, 0.6, 0.4]
MIN_INTERACTIONS = 5
EPS              = 1e-8


# ─────────────────────────────────────────────────────────────
# Step 1: 加载原始数据，构造 B_u
# ─────────────────────────────────────────────────────────────
print("[1] 加载原始数据，构造 B_u ...")
df = pd.read_csv(DATA_PATH, usecols=["user_id", "item_id", "rating"])
user_counts  = df.groupby("user_id").size()
valid_users  = user_counts[user_counts >= MIN_INTERACTIONS].index
df           = df[df["user_id"].isin(valid_users)].copy()
user_ids     = sorted(df["user_id"].unique())
item_ids     = sorted(df["item_id"].unique())
user2idx     = {u: i for i, u in enumerate(user_ids)}
item2idx     = {it: i for i, it in enumerate(item_ids)}
df["uid"]    = df["user_id"].map(user2idx)
df["iid"]    = df["item_id"].map(item2idx)
Bu           = df.groupby("uid")["iid"].apply(set).to_dict()
print(f"  总用户数: {len(Bu)}, 总物品数: {len(item_ids)}")


# ─────────────────────────────────────────────────────────────
# Step 2: 加载模型文件
# ─────────────────────────────────────────────────────────────
print("\n[2] 加载模型文件 ...")
ckpt          = torch.load(EMBED_PATH, map_location="cpu")
user_emb_orig = ckpt["user_embeddings"].numpy()
item_emb_orig = ckpt["item_embeddings"].numpy()

basis_ckpt        = torch.load(f"{BASIS_DIR}/pattern_basis.pt", map_location="cpu")
pattern_basis     = basis_ckpt["pattern_basis"].numpy()

sets_ckpt         = torch.load(f"{BASIS_DIR}/pattern_item_sets.pt", map_location="cpu")
pattern_item_sets = sets_ckpt["pattern_item_sets"]

infl_ckpt        = torch.load(INFLUENCE_PATH, map_location="cpu", weights_only=False)
influence_matrix = infl_ckpt["influence_matrix"]
sampled_uids     = infl_ckpt["sampled_uids"]
hq_patterns      = infl_ckpt["hq_patterns"]
K                = len(hq_patterns)

unlearn_ckpt  = torch.load(UNLEARN_PATH, map_location="cpu", weights_only=False)
user_emb_proj = unlearn_ckpt["user_emb_after"]

d = user_emb_orig.shape[1]
print(f"  sampled 用户数: {len(sampled_uids)}, HQ pattern 数: {K}, dim: {d}")


# ─────────────────────────────────────────────────────────────
# Step 3: 计算 S_u、I_target、I_non_target
# ─────────────────────────────────────────────────────────────
print("\n[3] 计算 S_u、I_target(u)、I_non_target(u) ...")
mu_u    = influence_matrix.mean(axis=1)
sigma_u = influence_matrix.std(axis=1)
tau_u   = mu_u + sigma_u

Su_dict           = {}
I_target_dict     = {}
I_non_target_dict = {}

for idx, uid in enumerate(sampled_uids):
    scores    = influence_matrix[idx]
    threshold = tau_u[idx]
    sorted_ki = np.argsort(scores)[::-1]
    targets   = [ki for ki in sorted_ki if scores[ki] > threshold]
    if len(targets) == 0:
        continue
    Su_dict[uid] = targets
    bu           = Bu.get(uid, set())
    union_Ik     = set()
    for ki in targets:
        union_Ik.update(pattern_item_sets[hq_patterns[ki]])
    I_tgt  = bu & union_Ik
    I_ntgt = bu - I_tgt
    if len(I_tgt) > 0:
        I_target_dict[uid]     = list(I_tgt)
        I_non_target_dict[uid] = list(I_ntgt)

U_T       = list(I_target_dict.keys())
UT_to_pos = {uid: pos for pos, uid in enumerate(U_T)}
print(f"  |U_T|: {len(U_T)}")
print(f"  平均 |I_target(u)|:     {np.mean([len(v) for v in I_target_dict.values()]):.2f}")
print(f"  平均 |I_non_target(u)|: {np.mean([len(v) for v in I_non_target_dict.values()]):.2f}")


# ─────────────────────────────────────────────────────────────
# Step 4: I_aff
# ─────────────────────────────────────────────────────────────
print("\n[4] 计算 I_aff ...")
I_aff     = sorted(set(iid for uid in U_T for iid in I_target_dict[uid]))
I_aff_set = set(I_aff)
iaff_to_pos = {iid: pos for pos, iid in enumerate(I_aff)}
print(f"  |I_aff|: {len(I_aff)}")


# ─────────────────────────────────────────────────────────────
# Step 5: 初始化优化变量
# ─────────────────────────────────────────────────────────────
print("\n[5] 初始化优化变量 ...")
e_u_tilde = torch.tensor(
    np.stack([user_emb_proj[uid] for uid in U_T]),
    dtype=torch.float32, requires_grad=True
)
delta_x = torch.zeros(len(I_aff), d, dtype=torch.float32, requires_grad=True)

item_emb_orig_t = torch.tensor(item_emb_orig, dtype=torch.float32)
user_emb_orig_t = torch.tensor(user_emb_orig, dtype=torch.float32)
pattern_basis_t = torch.tensor(pattern_basis, dtype=torch.float32)
x_iaff_orig_t   = item_emb_orig_t[I_aff]   # (|I_aff|, d)  固定


# ─────────────────────────────────────────────────────────────
# Step 6: 预计算索引张量（一次性，用于向量化 loss）
# ─────────────────────────────────────────────────────────────
print("\n[6] 预计算索引张量 ...")

# ── L_pattern 索引：(u_pos, pattern_global_k) ──────────────
pat_u_idx = []   # u_pos in U_T
pat_k_idx = []   # pattern global index k
for uid in U_T:
    u_pos = UT_to_pos[uid]
    for ki in Su_dict.get(uid, []):
        pat_u_idx.append(u_pos)
        pat_k_idx.append(hq_patterns[ki])
pat_u_idx = torch.tensor(pat_u_idx, dtype=torch.long)
pat_k_idx = torch.tensor(pat_k_idx, dtype=torch.long)

# ── L_target 索引 ──────────────────────────────────────────
tgt_u_idx = []   # u_pos
tgt_i_pos = []   # i_pos in I_aff
tgt_s_orig = []  # s_orig(u, i)
for uid in U_T:
    u_pos = UT_to_pos[uid]
    eu    = user_emb_orig[uid]
    for iid in I_target_dict[uid]:
        tgt_u_idx.append(u_pos)
        tgt_i_pos.append(iaff_to_pos[iid])
        tgt_s_orig.append(float(eu @ item_emb_orig[iid]))
tgt_u_idx  = torch.tensor(tgt_u_idx,  dtype=torch.long)
tgt_i_pos  = torch.tensor(tgt_i_pos,  dtype=torch.long)
tgt_s_orig = torch.tensor(tgt_s_orig, dtype=torch.float32)

# ── L_history 索引（分两类：in_aff / out_aff）─────────────
hist_in_u  = []; hist_in_i  = []; hist_in_s  = []
hist_out_u = []; hist_out_i = []; hist_out_s = []
for uid in U_T:
    u_pos = UT_to_pos[uid]
    eu    = user_emb_orig[uid]
    for iid in I_non_target_dict.get(uid, []):
        s = float(eu @ item_emb_orig[iid])
        if iid in I_aff_set:
            hist_in_u.append(u_pos); hist_in_i.append(iaff_to_pos[iid]); hist_in_s.append(s)
        else:
            hist_out_u.append(u_pos); hist_out_i.append(iid); hist_out_s.append(s)
hist_in_u  = torch.tensor(hist_in_u,  dtype=torch.long)
hist_in_i  = torch.tensor(hist_in_i,  dtype=torch.long)
hist_in_s  = torch.tensor(hist_in_s,  dtype=torch.float32)
hist_out_u = torch.tensor(hist_out_u, dtype=torch.long)
hist_out_i = torch.tensor(hist_out_i, dtype=torch.long)
hist_out_s = torch.tensor(hist_out_s, dtype=torch.float32)

# ── L_top 索引 ─────────────────────────────────────────────
top_in_u  = []; top_in_i  = []; top_in_s  = []
top_out_u = []; top_out_i = []; top_out_s = []
for uid in U_T:
    u_pos    = UT_to_pos[uid]
    eu       = user_emb_orig[uid]
    topk_set = set(np.argpartition(item_emb_orig @ eu, -TOP_K)[-TOP_K:])
    pat_items = set()
    for ki in Su_dict.get(uid, []):
        pat_items.update(pattern_item_sets[hq_patterns[ki]])
    R_top = topk_set - pat_items
    for iid in R_top:
        s = float(eu @ item_emb_orig[iid])
        if iid in I_aff_set:
            top_in_u.append(u_pos); top_in_i.append(iaff_to_pos[iid]); top_in_s.append(s)
        else:
            top_out_u.append(u_pos); top_out_i.append(iid); top_out_s.append(s)
top_in_u  = torch.tensor(top_in_u,  dtype=torch.long)
top_in_i  = torch.tensor(top_in_i,  dtype=torch.long)
top_in_s  = torch.tensor(top_in_s,  dtype=torch.float32)
top_out_u = torch.tensor(top_out_u, dtype=torch.long)
top_out_i = torch.tensor(top_out_i, dtype=torch.long)
top_out_s = torch.tensor(top_out_s, dtype=torch.float32)

# ── L_collateral 索引 ──────────────────────────────────────
non_target_uid_set = set(U_T)
collat_eu  = []   # e_u_orig 向量（直接存，避免重复索引）
collat_i   = []   # i_pos in I_aff
for uid in sampled_uids:
    if uid in non_target_uid_set:
        continue
    bu = Bu.get(uid, set())
    for iid in bu:
        if iid in I_aff_set:
            collat_eu.append(user_emb_orig[uid])
            collat_i.append(iaff_to_pos[iid])
has_collat = len(collat_eu) > 0
collat_eu = torch.tensor(np.array(collat_eu), dtype=torch.float32) if has_collat else None
collat_i  = torch.tensor(collat_i, dtype=torch.long)              if has_collat else None

# ── L_user_reg：u_orig 向量 & 范数 ────────────────────────
eu_orig_ut      = user_emb_orig_t[[uid for uid in U_T]]   # (|U_T|, d)
eu_orig_norm_sq = (eu_orig_ut ** 2).sum(dim=1) + EPS       # (|U_T|,)

# ── item orig 范数（for L_item_reg，已在 Step 5 有 x_iaff_orig_t）─
x_orig_norm_sq = (x_iaff_orig_t ** 2).sum(dim=1) + EPS     # (|I_aff|,)

print(f"  L_pattern pairs:    {len(pat_u_idx)}")
print(f"  L_target pairs:     {len(tgt_u_idx)}")
print(f"  L_history pairs:    {len(hist_in_u) + len(hist_out_u)}")
print(f"  L_top pairs:        {len(top_in_u) + len(top_out_u)}")
print(f"  L_collateral pairs: {len(collat_i) if collat_i is not None else 0}")


# ─────────────────────────────────────────────────────────────
# Step 7: 向量化优化循环
# ─────────────────────────────────────────────────────────────
print("\n[7] 开始 Local Repair 优化（向量化）...")
optimizer = torch.optim.Adam([e_u_tilde, delta_x], lr=LR)

for epoch in range(1, EPOCHS + 1):
    optimizer.zero_grad()

    x_iaff_rep = x_iaff_orig_t + delta_x   # (|I_aff|, d)

    # ── L_pattern ────────────────────────────────────────────
    # (e_u_tilde[u_pos] · v_k)^2，向量化
    eu_pat = e_u_tilde[pat_u_idx]                        # (P, d)
    vk_pat = pattern_basis_t[pat_k_idx]                  # (P, d)
    proj   = (eu_pat * vk_pat).sum(dim=1)                # (P,)
    loss_pattern = (proj ** 2).mean()

    # ── L_target ─────────────────────────────────────────────
    eu_tgt  = e_u_tilde[tgt_u_idx]                       # (T, d)
    xi_tgt  = x_iaff_rep[tgt_i_pos]                      # (T, d)
    s_rep   = (eu_tgt * xi_tgt).sum(dim=1)               # (T,)
    thresh  = tgt_s_orig - MARGIN
    loss_target = (TEMPERATURE * F.softplus(
        (s_rep - thresh) / TEMPERATURE
    )).mean()

    # ── L_history ────────────────────────────────────────────
    hist_terms = []
    if len(hist_in_u) > 0:
        eu_h  = e_u_tilde[hist_in_u]
        xi_h  = x_iaff_rep[hist_in_i]
        s_h   = (eu_h * xi_h).sum(dim=1)
        hist_terms.append(((s_h - hist_in_s) ** 2))
    if len(hist_out_u) > 0:
        eu_h  = e_u_tilde[hist_out_u]
        xi_h  = item_emb_orig_t[hist_out_i]
        s_h   = (eu_h * xi_h).sum(dim=1)
        hist_terms.append(((s_h - hist_out_s) ** 2))
    loss_history = torch.cat(hist_terms).mean() if hist_terms else torch.tensor(0.0)

    # ── L_top ────────────────────────────────────────────────
    top_terms = []
    if len(top_in_u) > 0:
        eu_t  = e_u_tilde[top_in_u]
        xi_t  = x_iaff_rep[top_in_i]
        s_t   = (eu_t * xi_t).sum(dim=1)
        top_terms.append(((s_t - top_in_s) ** 2))
    if len(top_out_u) > 0:
        eu_t  = e_u_tilde[top_out_u]
        xi_t  = item_emb_orig_t[top_out_i]
        s_t   = (eu_t * xi_t).sum(dim=1)
        top_terms.append(((s_t - top_out_s) ** 2))
    loss_top = torch.cat(top_terms).mean() if top_terms else torch.tensor(0.0)

    # ── L_collateral ─────────────────────────────────────────
    if collat_eu is not None and len(collat_eu) > 0:
        dx_c        = delta_x[collat_i]                  # (C, d)
        dot_c       = (collat_eu * dx_c).sum(dim=1)      # (C,)
        loss_collat = (dot_c ** 2).mean()
    else:
        loss_collat = torch.tensor(0.0)

    # ── L_user_reg ───────────────────────────────────────────
    diff_u        = e_u_tilde - eu_orig_ut               # (|U_T|, d)
    loss_user_reg = ((diff_u ** 2).sum(dim=1) / eu_orig_norm_sq).mean()

    # ── L_item_reg ───────────────────────────────────────────
    loss_item_reg = ((delta_x ** 2).sum(dim=1) / x_orig_norm_sq).mean()

    # ── 合并 ─────────────────────────────────────────────────
    loss = (W_PATTERN  * loss_pattern  +
            W_TARGET   * loss_target   +
            W_HISTORY  * loss_history  +
            W_TOP      * loss_top      +
            W_COLLAT   * loss_collat   +
            W_USER_REG * loss_user_reg +
            W_ITEM_REG * loss_item_reg)

    loss.backward()
    optimizer.step()

    if epoch % LOG_EVERY == 0 or epoch == 1:
        print(f"  Epoch {epoch:>4d}/{EPOCHS} | loss={loss.item():.4f} | "
              f"L_pat={loss_pattern.item():.4f} | L_tgt={loss_target.item():.4f} | "
              f"L_hist={loss_history.item():.4f} | L_top={loss_top.item():.4f} | "
              f"L_col={loss_collat.item():.4f} | "
              f"L_ureg={loss_user_reg.item():.4f} | L_ireg={loss_item_reg.item():.4f}")

print("  优化完成")


# ─────────────────────────────────────────────────────────────
# Step 8: 提取 repaired embeddings
# ─────────────────────────────────────────────────────────────
print("\n[8] 提取 repaired embeddings ...")
e_u_rep_np = e_u_tilde.detach().numpy()
delta_x_np = delta_x.detach().numpy()

user_emb_repaired = user_emb_orig.copy()
for pos, uid in enumerate(U_T):
    user_emb_repaired[uid] = e_u_rep_np[pos]

item_emb_repaired = item_emb_orig.copy()
for pos, iid in enumerate(I_aff):
    item_emb_repaired[iid] = item_emb_orig[iid] + delta_x_np[pos]


# ─────────────────────────────────────────────────────────────
# Step 9: 评估
# ─────────────────────────────────────────────────────────────
print("\n[9] 评估指标 ...")

pfr_proj_list    = []; pfr_repair_list    = []
overlap_proj_list = []; overlap_repair_list = []
tsd_proj_list    = []; tsd_repair_list    = []
ntsc_proj_list   = []; ntsc_repair_list   = []

for uid in U_T:
    eu_orig   = user_emb_orig[uid]
    eu_proj   = user_emb_proj[uid]
    eu_repair = user_emb_repaired[uid]
    ki_list   = Su_dict.get(uid, [])

    # PFR
    A_b_proj = sum(abs(eu_orig   @ pattern_basis[hq_patterns[ki]]) for ki in ki_list)
    A_a_proj = sum(abs(eu_proj   @ pattern_basis[hq_patterns[ki]]) for ki in ki_list)
    A_a_rep  = sum(abs(eu_repair @ pattern_basis[hq_patterns[ki]]) for ki in ki_list)
    pfr_proj_list.append(1.0 - A_a_proj / (A_b_proj + EPS) if A_b_proj > 0 else 0.0)
    pfr_repair_list.append(1.0 - A_a_rep  / (A_b_proj + EPS) if A_b_proj > 0 else 0.0)

    # Overlap@K with Original
    topk_orig   = set(np.argpartition(item_emb_orig     @ eu_orig,   -TOP_K)[-TOP_K:])
    topk_proj   = set(np.argpartition(item_emb_orig     @ eu_proj,   -TOP_K)[-TOP_K:])
    topk_repair = set(np.argpartition(item_emb_repaired @ eu_repair, -TOP_K)[-TOP_K:])
    overlap_proj_list.append(len(topk_orig & topk_proj)   / TOP_K)
    overlap_repair_list.append(len(topk_orig & topk_repair) / TOP_K)

    # Target Score Drop
    I_tgt = I_target_dict[uid]
    if I_tgt:
        s_o = np.array([eu_orig   @ item_emb_orig[i]    for i in I_tgt])
        s_p = np.array([eu_proj   @ item_emb_orig[i]    for i in I_tgt])
        s_r = np.array([eu_repair @ item_emb_repaired[i] for i in I_tgt])
        tsd_proj_list.append((s_o - s_p).mean())
        tsd_repair_list.append((s_o - s_r).mean())

    # Non-target Score Change
    I_ntgt = I_non_target_dict.get(uid, [])
    if I_ntgt:
        s_o = np.array([eu_orig   @ item_emb_orig[i]    for i in I_ntgt])
        s_p = np.array([eu_proj   @ item_emb_orig[i]    for i in I_ntgt])
        s_r = np.array([eu_repair @ item_emb_repaired[i] for i in I_ntgt])
        ntsc_proj_list.append(np.abs(s_o - s_p).mean())
        ntsc_repair_list.append(np.abs(s_o - s_r).mean())

print(f"\n  评估用户数: {len(U_T)}")
print(f"\n  {'指标':<32} {'Projection':>12} {'Repair':>12}")
print(f"  {'-'*56}")
print(f"  {'PFR_multi (mean)':<32} {np.mean(pfr_proj_list):>12.4f} {np.mean(pfr_repair_list):>12.4f}")
print(f"  {'Overlap@K w/ Orig (mean)':<32} {np.mean(overlap_proj_list):>12.4f} {np.mean(overlap_repair_list):>12.4f}")
print(f"  {'Target Score Drop (mean)':<32} {np.mean(tsd_proj_list):>12.4f} {np.mean(tsd_repair_list):>12.4f}")
print(f"  {'Non-target Score Change (mean)':<32} {np.mean(ntsc_proj_list):>12.4f} {np.mean(ntsc_repair_list):>12.4f}")


# ─────────────────────────────────────────────────────────────
# Step 10: 保存
# ─────────────────────────────────────────────────────────────
print(f"\n[10] 保存结果至 {SAVE_DIR} ...")
import torch as _torch
_torch.save(
    {"user_emb_repaired": user_emb_repaired,
     "item_emb_repaired": item_emb_repaired,
     "delta_x":           delta_x_np,
     "U_T":               U_T,
     "I_aff":             I_aff},
    os.path.join(SAVE_DIR, "embeddings_repaired.pt")
)
_torch.save(
    {"pfr_proj":        pfr_proj_list,    "pfr_repair":      pfr_repair_list,
     "overlap_proj":    overlap_proj_list, "overlap_repair":  overlap_repair_list,
     "tsd_proj":        tsd_proj_list,    "tsd_repair":      tsd_repair_list,
     "ntsc_proj":       ntsc_proj_list,   "ntsc_repair":     ntsc_repair_list,
     "hyperparams": {
         "W_PATTERN": W_PATTERN, "W_TARGET": W_TARGET,
         "W_HISTORY": W_HISTORY, "W_TOP":    W_TOP,
         "W_COLLAT":  W_COLLAT,  "W_USER_REG": W_USER_REG, "W_ITEM_REG": W_ITEM_REG,
         "MARGIN": MARGIN, "TEMPERATURE": TEMPERATURE,
         "LR": LR, "EPOCHS": EPOCHS}},
    os.path.join(SAVE_DIR, "eval_results_repair.pt")
)
print("  embeddings_repaired.pt  — repaired user/item embeddings")
print("  eval_results_repair.pt  — Projection vs Repair 对比指标")
print("\n[完成]")

[1] 加载原始数据，构造 B_u ...
  总用户数: 27794, 总物品数: 144853

[2] 加载模型文件 ...
  sampled 用户数: 1000, HQ pattern 数: 8, dim: 64

[3] 计算 S_u、I_target(u)、I_non_target(u) ...
  |U_T|: 889
  平均 |I_target(u)|:     4.16
  平均 |I_non_target(u)|: 8.34

[4] 计算 I_aff ...
  |I_aff|: 2144

[5] 初始化优化变量 ...

[6] 预计算索引张量 ...
  L_pattern pairs:    1341
  L_target pairs:     3696
  L_history pairs:    7412
  L_top pairs:        2059
  L_collateral pairs: 48

[7] 开始 Local Repair 优化（向量化）...
  Epoch    1/200 | loss=2.5915 | L_pat=0.0037 | L_tgt=0.2962 | L_hist=1.4357 | L_top=0.5605 | L_col=0.0000 | L_ureg=0.1187 | L_ireg=0.0000
  Epoch   20/200 | loss=0.2233 | L_pat=0.0327 | L_tgt=0.0110 | L_hist=0.0821 | L_top=0.0220 | L_col=0.0090 | L_ureg=0.1847 | L_ireg=0.0503
  Epoch   40/200 | loss=0.0946 | L_pat=0.0170 | L_tgt=0.0123 | L_hist=0.0255 | L_top=0.0032 | L_col=0.0010 | L_ureg=0.1947 | L_ireg=0.0258
  Epoch   60/200 | loss=0.0705 | L_pat=0.0142 | L_tgt=0.0119 | L_hist=0.0142 | L_top=0.0009 | L_col=0.0002 | L_ureg=0.1996 